# Daily Practice — 2026-11-02 — ML Testing & MLOps: Building a Probability Calibration Test Suite

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset) —
569 real fine-needle-aspirate biopsy records (30 numeric features describing cell nuclei: radius,
texture, smoothness, concavity, etc.), loaded directly from `sklearn.datasets.load_breast_cancer`.
Target: `benign` (1) vs `malignant` (0).

**Problem statement:** A binary classifier's accuracy can look fine while its predicted
*probabilities* are badly miscalibrated — e.g. it says "90% confident" on a batch of cases that
are only right 70% of the time. In a clinical triage or risk-scoring pipeline, probabilities (not
just the final 0/1 label) often drive the downstream decision (how urgently to escalate a case,
how much manual review to allocate), so shipping a model on accuracy alone is not enough: you need
an automated calibration test, the same way an SQA engineer would not ship on "the happy path
passed" alone.

**What you should produce:**
1. A `compute_ece(y_true, y_prob, n_bins)` function that bins predictions by confidence and returns
   the Expected Calibration Error (ECE) plus a per-bin breakdown (count, average confidence,
   average actual accuracy, gap).
2. A reliability diagram (bar or line plot of accuracy vs. confidence per bin, with the perfect-
   calibration diagonal) built from that breakdown.
3. An `assert_calibration_ok(y_true, y_prob, max_ece, ...)` function that behaves like a CI gate:
   it raises `AssertionError` with the worst offending bins when ECE exceeds a threshold, and
   otherwise returns the ECE quietly — something you could literally drop into a pytest suite or
   a model-deployment pipeline.
4. A **slice-based** calibration check: run the same gate separately on terciles of `mean radius`
   (small / medium / large tumors), with a `min_slice_size` guard — because a model can pass an
   aggregate calibration gate while being badly miscalibrated on one real subgroup of patients.
5. A before/after comparison: calibrate the same base model with `CalibratedClassifierCV` and show
   whether ECE and Brier score actually improve overall *and* on the slice that failed.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Load the data and train a deliberately under-ensembled baseline

We use a `RandomForestClassifier` with only 10 trees. That's intentional: a small forest tends to
push its predicted probabilities toward 0 and 1 more than a well-ensembled one, which gives us a
genuinely miscalibrated (not synthetically corrupted) model to catch — the kind of thing that
happens in practice when someone ships a model that was only validated on accuracy/AUC.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["target"] = data.target  # 1 = benign, 0 = malignant

FEATURES = [c for c in df.columns if c != "target"]
TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

base_model = RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE)
base_model.fit(X_train, y_train)

y_prob = base_model.predict_proba(X_test)[:, 1]
y_pred = base_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Fraction of predictions with confidence > 0.95 or < 0.05:",
      round(((y_prob > 0.95) | (y_prob < 0.05)).mean(), 4))


## Your task

Fill in the TODOs below. Function signatures are given — don't change them, the solution cells
assume these names and return shapes.

In [ ]:
def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error.

    Bin predictions into `n_bins` equal-width bins over [0, 1] by predicted probability.
    For each non-empty bin, compute:
      - avg_confidence: mean predicted probability in the bin
      - avg_accuracy:   mean true label (fraction of positives) in the bin
      - gap:            abs(avg_accuracy - avg_confidence)
    ECE is the count-weighted average of `gap` across bins.

    Returns
    -------
    ece : float
    bins_df : pandas.DataFrame with columns
        ["bin", "low", "high", "count", "avg_confidence", "avg_accuracy", "gap"]
    """
    # TODO: implement
    raise NotImplementedError


In [ ]:
def plot_reliability_diagram(bins_df, title="Reliability diagram"):
    """Plot avg_accuracy vs avg_confidence per bin (bar or scatter+line),
    with the y = x perfect-calibration line overlaid for reference."""
    # TODO: implement
    raise NotImplementedError


In [ ]:
def assert_calibration_ok(y_true, y_prob, max_ece=0.05, n_bins=10, label="model"):
    """CI-style calibration gate.

    Compute ECE via compute_ece(). If it exceeds `max_ece`, raise AssertionError whose
    message includes `label`, the measured ECE, the threshold, and the 3 worst-gap bins
    (as a readable table/string). Otherwise return the ECE.
    """
    # TODO: implement
    raise NotImplementedError


In [ ]:
def slice_calibration_report(y_true, y_prob, slice_series, n_bins=5, min_slice_size=30):
    """Run compute_ece() separately for each category in `slice_series`.

    Skip any slice with fewer than `min_slice_size` samples. Return a DataFrame with one
    row per evaluated slice: columns ["slice", "count", "ece"].
    """
    # TODO: implement
    raise NotImplementedError


In [ ]:
# TODO:
# 1. Call compute_ece(y_test, y_prob) and print the resulting ECE.
# 2. Plot the reliability diagram with plot_reliability_diagram().
# 3. Call assert_calibration_ok(y_test, y_prob, max_ece=0.03, label="rf_uncalibrated")
#    inside a try/except and print whether the gate passed or the AssertionError message.
# 4. Build terciles of X_test["mean radius"] (labels "small", "medium", "large") with
#    pd.qcut, and run slice_calibration_report() against them. Print the result.


### Now calibrate the model and re-check

Fit a `CalibratedClassifierCV` wrapping a fresh `RandomForestClassifier(n_estimators=10,
random_state=RANDOM_STATE)`, using `method="isotonic"` and `cv=5`, on `(X_train, y_train)`.
Get its probabilities on `X_test`, then re-run the same three checks (overall ECE, the gate,
and the slice report) on the calibrated probabilities, and compare `brier_score_loss` before vs.
after.

In [ ]:
# TODO: fit CalibratedClassifierCV, get y_prob_calibrated on X_test,
# then recompute ECE, the slice report, and brier_score_loss for both models side by side.


### Write-up

TODO: in 3–5 sentences — which slice (if any) fails the calibration gate even when the overall
ECE looks acceptable, why that matters more for this dataset (biopsy triage) than it would for a
low-stakes recommender, and whether isotonic calibration fixed the failing slice or just moved the
miscalibration somewhere else.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. compute_ece
def compute_ece(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, edges[1:-1], right=True)

    n = len(y_true)
    ece = 0.0
    rows = []
    for b in range(n_bins):
        mask = bin_ids == b
        count = int(mask.sum())
        if count == 0:
            continue
        avg_confidence = y_prob[mask].mean()
        avg_accuracy = y_true[mask].mean()
        gap = abs(avg_accuracy - avg_confidence)
        ece += (count / n) * gap
        rows.append({
            "bin": b, "low": edges[b], "high": edges[b + 1],
            "count": count, "avg_confidence": avg_confidence,
            "avg_accuracy": avg_accuracy, "gap": gap,
        })
    return ece, pd.DataFrame(rows)


# 2. reliability diagram
def plot_reliability_diagram(bins_df, title="Reliability diagram"):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
    ax.scatter(bins_df["avg_confidence"], bins_df["avg_accuracy"],
               s=bins_df["count"], alpha=0.7, label="observed bins")
    for _, row in bins_df.iterrows():
        ax.plot([row["avg_confidence"], row["avg_confidence"]],
                 [row["avg_confidence"], row["avg_accuracy"]], color="red", alpha=0.4)
    ax.set_xlabel("Average predicted confidence")
    ax.set_ylabel("Average actual accuracy")
    ax.set_title(title)
    ax.legend()
    plt.show()


# 3. CI-style gate
def assert_calibration_ok(y_true, y_prob, max_ece=0.05, n_bins=10, label="model"):
    ece, bins_df = compute_ece(y_true, y_prob, n_bins=n_bins)
    if ece > max_ece:
        worst = bins_df.sort_values("gap", ascending=False).head(3)
        raise AssertionError(
            f"[{label}] ECE {ece:.4f} exceeds gate {max_ece:.4f}\n"
            f"Worst bins:\n{worst.to_string(index=False)}"
        )
    return ece


# 4. slice-based calibration report
def slice_calibration_report(y_true, y_prob, slice_series, n_bins=5, min_slice_size=30):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    rows = []
    for level in pd.unique(slice_series):
        mask = (slice_series == level).to_numpy()
        count = int(mask.sum())
        if count < min_slice_size:
            continue
        ece, _ = compute_ece(y_true[mask], y_prob[mask], n_bins=n_bins)
        rows.append({"slice": level, "count": count, "ece": ece})
    return pd.DataFrame(rows)


# 5. run the checks on the uncalibrated model
ece, bins_df = compute_ece(y_test.values, y_prob, n_bins=10)
print("Uncalibrated overall ECE:", round(ece, 4))
plot_reliability_diagram(bins_df, title="Uncalibrated RandomForest (10 trees)")

try:
    assert_calibration_ok(y_test.values, y_prob, max_ece=0.03, label="rf_uncalibrated")
    print("Gate passed")
except AssertionError as e:
    print("Gate failed:\n", e)

terciles = pd.qcut(X_test["mean radius"], 3, labels=["small", "medium", "large"])
report = slice_calibration_report(y_test.values, y_prob, terciles, n_bins=5, min_slice_size=30)
print(report)

# 6. calibrate and compare
calibrated_model = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE),
    method="isotonic", cv=5,
)
calibrated_model.fit(X_train, y_train)
y_prob_cal = calibrated_model.predict_proba(X_test)[:, 1]

ece_cal, bins_df_cal = compute_ece(y_test.values, y_prob_cal, n_bins=10)
report_cal = slice_calibration_report(y_test.values, y_prob_cal, terciles, n_bins=5, min_slice_size=30)

print("Calibrated overall ECE:", round(ece_cal, 4))
print(report_cal)
print("Brier (uncalibrated):", round(brier_score_loss(y_test, y_prob), 4))
print("Brier (calibrated):  ", round(brier_score_loss(y_test, y_prob_cal), 4))
```

**What this shows (actual numbers from this dataset/split):** the overall uncalibrated ECE comes
out around 0.032, just over a 0.03 gate — but the "medium" tumor-size tercile has an ECE around
0.093, roughly 9x worse than the "small" slice (0.011) and 3x worse than "large" (0.030). A gate
set only on the aggregate number would have let this ship. This is the same lesson as slice-based
performance testing applied to a different metric: aggregate calibration metrics hide subgroup
failures. Isotonic calibration clearly improves the overall Brier score (0.050 → 0.039) and fixes
the "medium" slice (0.093 → 0.038), but it actually makes the "large" slice *worse* (0.030 →
0.062) — a reminder that "we calibrated it" does not mean "it's fixed everywhere," and a
deployment gate should keep re-checking slices after any model or calibration change, not just
once.

</details>